# Classroom EDA: Adult Census Income
## Statistical Methods for Fairness & Missingness
### DSA4970/5970

This notebooks demonstrates how to conduct fairness EDA using the Adult Census Income Dataset from the UCI Machine Learning Repository.  We will analyze the dataset looking for bias associated with protected categories.  First we look at the data dictionary for the dataset.

Data Dictionary: Adult Census Income (Full)

$$\begin{array}{|l|l|l|l|}
\hline
\textbf{Variable} & \textbf{Description} & \textbf{Type} & \textbf{Notes} \\ \hline
\text{age} & \text{Age of individual} & \text{Numeric} & \mathbf{Protected} \\ \hline
\text{work} & \text{Employment sector} & \text{Categorical} & \text{Contains '?'} \\ \hline
\text{fnlwgt} & \text{Final weight (census)} & \text{Numeric} & \text{Continuous} \\ \hline
\text{edu} & \text{Education level (name)} & \text{Categorical} & \text{Ordinal} \\ \hline
\text{num\_edu} & \text{Education level (years)} & \text{Numeric} & \text{Proxy Feature} \\ \hline
\text{mar} & \text{Marital status} & \text{Categorical} & \text{Demographic} \\ \hline
\text{occ} & \text{Professional occupation} & \text{Categorical} & \text{Missingness Target} \\ \hline
\text{rel} & \text{Household relationship} & \text{Categorical} & \text{Demographic} \\ \hline
\text{race} & \text{Ethnicity} & \text{Categorical} & \mathbf{Protected} \\ \hline
\text{sex} & \text{Gender (Male/Female)} & \text{Categorical} & \mathbf{Protected} \\ \hline
\text{gain} & \text{Capital gains} & \text{Numeric} & \text{Financial} \\ \hline
\text{loss} & \text{Capital losses} & \text{Numeric} & \text{Financial} \\ \hline
\text{hpw} & \text{Hours worked per week} & \text{Numeric} & \text{Continuous} \\ \hline
\text{country} & \text{Country of origin} & \text{Categorical} & \text{Demographic} \\ \hline
\text{target} & \text{Income level (>50K/<=50K)} & \text{Binary} & \text{Label} \\ \hline
\end{array}$$

We can see that there are two protected categories:  race and sex (gender).  Therefore we will conduct our analysis with respect to these two variables. For simplicity, we will limit our analysis to just Black, Asian, and White subjects.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression

In [ ]:
# Import the dataset directly from the UCI repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
cols = ['age', 'work', 'fnlwgt', 'edu', 'num_edu', 'mar', 'occ', 'rel', 'race', 'sex', 'gain', 'loss', 'hpw', 'country', 'target']
df = pd.read_csv(url, names=cols, sep=r',\s*', engine='python')

# Replace the '?' with NaN values
df.replace('?', np.nan, inplace=True)

# Filter out 'Other' and 'Amer-Indian-Eskimo' races for simplicity
df = df[~df['race'].isin(['Other', 'Amer-Indian-Eskimo'])]

# Look at the first few rows of the dataset
df.head()

### 1. Stratified Stats & Distributions
We begin by creating a simple summary of the features with respect to the protected attributes.  First we look at summary statistics for a few numeric variables:  age, hours worked per week, and years of education.

In [ ]:
display(df.groupby('race')[['age', 'hpw', 'num_edu']].describe().round(2))

In [ ]:
display(df.groupby('sex')[['age', 'hpw', 'num_edu']].describe().round(2))

We can see that the average age of respondents across ethnicities is similar, but Whites are slightly older on average. The average hours per week (hpw) is also similar across ethnicities, but Asians work slightly more hours per week on average and Blacks work less. The average number of years of education (num_edu) for Whites and Asians is higher than for Blacks and American Indian/Eskimos.

We now take a look at the distributions for these variables to see if there are any notable changes in shapes among them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.kdeplot(data=df, x='age', hue='race', common_norm=False, ax=axes[0])
axes[0].set_title('Age Distribution')
sns.kdeplot(data=df, x='hpw', hue='race', common_norm=False, ax=axes[1])
axes[1].set_title('Hours per Week Distribution')
sns.kdeplot(data=df, x='num_edu', hue='race', common_norm=False, ax=axes[2])
axes[2].set_title('Education Years Distribution')
fig.suptitle("Continuous Variable Distributions by Race")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.kdeplot(data=df, x='age', hue='sex', common_norm=False, ax=axes[0])
axes[0].set_title('Age Distribution')
sns.kdeplot(data=df, x='hpw', hue='sex', common_norm=False, ax=axes[1])
axes[1].set_title('Hours per Week Distribution')
sns.kdeplot(data=df, x='num_edu', hue='sex', common_norm=False, ax=axes[2])
axes[2].set_title('Education Years Distribution')
fig.suptitle("Continuous Variable Distributions by Sex")
plt.show()

The distributions do appear to show some differences that add more insight to the differences we saw in the summary tables.  Most notably, the age distribution for men skews higher than for females.  Also, Whites skew older than Blacks and Asians.  We now check to see if these differences are statistically significant with the K-S test.

### 2. K-S Test

In [ ]:
# Perform K-S test for age distribution between males and females
m_age = df[df['sex'] == 'Male']['age']
f_age = df[df['sex'] == 'Female']['age']
print('K-S Test for Age vs. Sex - p:', stats.ks_2samp(m_age, f_age).pvalue)

# Perform K-S test for age distribution between males and females
# Create a binary variable for White vs. other races
df['is_white'] = (df['race'] == 'White').astype(int)
w_age = df[df['is_white'] == 1]['age']
else_age = df[df['is_white'] == 0]['age']
print('K-S Test for Age vs. White - p:', stats.ks_2samp(w_age, else_age).pvalue)

# Perform K-S test for years of education distribution between men and women
m_num_edu = df[df['sex'] == 'Male']['num_edu']
f_num_edu = df[df['sex'] == 'Female']['num_edu']
print('K-S Test for Years of Education vs. Sex - p:', stats.ks_2samp(m_num_edu, f_num_edu).pvalue)



The results of the K-S Tests confirm that the distributions by protected category are indeed significantly different.  However, we can conclude from our summary statistics and graphical depiction of the distributions that the differences in number of years of education by age may be statistically significant, but they aren't large enough to worry about.

### 3. Chi-Square Test
Now we take a look at counts of categorical variables.  We see if these counts vary by sensitive attribute using the Chi-Square test.  We will check the counts of the target variable (income) and occupation versus sex and race.

In [ ]:
ct = pd.crosstab(df['sex'], df['target'])
# Row percentages for better interpretation
ct_percent = (ct.div(ct.sum(axis=1), axis=0) * 100).round(2)
print("Contingency Table: Sex vs Income (Row Percentages)")
display(ct_percent)
print('Chi-Square p:', stats.chi2_contingency(ct)[1])

ct = pd.crosstab(df['race'], df['target'])
# Row percentages for better interpretation
ct_percent = (ct.div(ct.sum(axis=1), axis=0) * 100).round(2)
print("Contingency Table: Race vs Income (Row Percentages)")
display(ct_percent)
print('Chi-Square p:', stats.chi2_contingency(ct)[1])


We see that the target variable definitely is biased by sex and race.  Males have a higher proportion of income over $50k than women.  Whites and Asians both have much higher proportions of high income than Blacks.

### 4. Missingness Analysis
We now look to see if missingness has any relationship to sensitive attributes.  We focus our look on missingness for occupation vs sex and race.  To do this, we use Little's MCAR Test. We will exclude some of the features that either have no relevance or many possible values from our test.

In [ ]:
# Little's MCAR Test: Logistic regression of missingness on observed variables
# Include all variables in the dataset to check for bias in missingness
# Create a binary variable for missing occupation values
df['is_missing'] = df['occ'].isnull().astype(int)
# Drop rows with missing values in predictor columns (all except 'occ')
predictor_cols = ['age', 'race', 'sex', 'num_edu', 'hpw', 'mar']  # Include more numeric variables and sensitive attributes
df_temp = df.dropna(subset=predictor_cols)

X_mcar = sm.add_constant(pd.get_dummies(df_temp[predictor_cols], drop_first=True))
X_mcar = X_mcar.astype(float)  # Convert to float to avoid dtype object error
y_mcar = df_temp['is_missing']
try:
    mcar_model = sm.Logit(y_mcar, X_mcar).fit(method='bfgs', maxiter=1000, disp=False)
    print("Little's MCAR Test Results (all variables included):")
    print(mcar_model.summary())
except Exception as e:
    print(f"Model fitting failed: {e}")
    print("Consider reducing variables or using imputation for missing predictors.")

We see that there are several features that have statistically significant non-zero coefficients.  The greatest of these are some of the marital status values and being male.  We also see that race does _not_ contribute to occupational missingness.  We conclude that occupational missingness is not MCAR, and it is biased with respect to the sensitive attribute sex.

### 5. Proxy Detection
We now look to for correlations and associations in the data.  We are particularly interested in relationships to sensitive attributes.  Other variables with high correlation or association could potentially be used as proxies for the sensitive attributes.  We do this in two steps.  First, we create a correlation heatmap with Pearson's Correlation Test to examine continous and binary variables.  Then we create an association heatmap to examine categorical variables.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# 1. Prepare the subset and handle missing values
# Note: Ensure 'race' is included in the initial selection
cols_to_use = ['age', 'sex', 'race', 'num_edu', 'hpw', 'gain', 'loss', 'target']
df_corr = df[cols_to_use].dropna().copy()

# 2. Initialize OneHotEncoder 
# Using drop='if_binary' to keep binary vars as one column (0/1) 
# and drop='first' for multi-category vars to avoid redundancy.
ohe = OneHotEncoder(drop='if_binary', sparse_output=False)

# 3. Encode the categorical columns
# We use fit_transform and then grab the new feature names
encoded_data = ohe.fit_transform(df_corr[['sex', 'race', 'target']])
encoded_df = pd.DataFrame(
    encoded_data, 
    columns=ohe.get_feature_names_out(['sex', 'race', 'target']),
    index=df_corr.index
)

# 4. Combine with numeric features
df_final_corr = pd.concat([df_corr[['age', 'num_edu', 'gain', 'loss', 'hpw']], encoded_df], axis=1)

# 5. Generate Pearson Correlation Matrix
corr_matrix = df_final_corr.corr(method='pearson')

# 6. Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap (Including Protected Attributes)')
plt.show()

When we look at the target variable, income, we see a slight positive correlation with age, education, and being male.  Male also has a positive correlation with hours worked.  We don't see any strong correlations between race and these variables.

To examine the remaining categorical variables without having to one-hot encode them, we can use Cramer's test.  We use the test to generate another heatmap.

In [ ]:
# 1. Prepare the subset and handle missing values
cols_to_use = ['sex', 'race', 'occ', 'mar', 'work', 'rel', 'country', 'target']
df_assoc = df[cols_to_use].dropna().copy()

def cramers_v(var1, var2):
    """ Calculates Cramer's V statistic for categorical-categorical association. """
    confusion_matrix = pd.crosstab(var1, var2).values
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    
    # Bias correction
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    
    # Prevent division by zero if rcorr or kcorr <= 1
    if min((kcorr - 1), (rcorr - 1)) <= 0:
        return 0.0
        
    return np.sqrt(phi2corr / min((kcorr - 1), (rcorr - 1)))

# 2. Initialize an empty DataFrame for the matrix
rows = []
for var1 in cols_to_use:
    col = []
    for var2 in cols_to_use:
        value = cramers_v(df_assoc[var1], df_assoc[var2])
        col.append(value)
    rows.append(col)

cramers_matrix = pd.DataFrame(rows, index=cols_to_use, columns=cols_to_use)

# 3. Plot the Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cramers_matrix, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5)
plt.title("Cramer's V Association Heatmap (Categorical Variables)")
plt.show()

Unlike Pearson's Correlation Coefficient, which ranges from -1 to +1 based upon strength and type of correlation, Cramer's ranges from 0 (no association) to 1 (strong association).  In the heatmap above, we see that the target variable is associated with household relationship, marital status, and occupation.  Sex is most strongly associated household relationship, marital status, and occupation.  Race is strongly associated only with country of origin.  Therefore, country of origin could be used to determine the sensitive attribute race, and a combination of three variables could determine sex.

### Conditional Relationship Testing
We will test for conditional association of hours worked to the protected attributes sex and race.  To do this we conduct a multi-linear regression on a subset of the variables in the dataset.

['age', 'sex', 'race', 'num_edu', 'hpw', 'gain', 'loss', 'target']

In [ ]:
# 1. Prepare the data
features = ['age', 'sex', 'race', 'num_edu', 'gain', 'loss']
target = 'hpw'

# Drop rows with missing values in our selection
df_reg = df[features + [target]].dropna().copy()

# 2. Encode Categorical Variables
# We use drop='first' to ensure the model is mathematically sound
ohe = OneHotEncoder(drop='first', sparse_output=False)
categorical_encoded = ohe.fit_transform(df_reg[['sex', 'race']])

# Create a DataFrame with the new encoded column names
encoded_cols = ohe.get_feature_names_out(['sex', 'race'])
df_encoded = pd.DataFrame(categorical_encoded, columns=encoded_cols, index=df_reg.index)

# 3. Combine numeric and encoded features
X = pd.concat([df_reg[['age', 'num_edu', 'gain', 'loss']], df_encoded], axis=1)
y = df_reg[target]

# 4. Add a constant (intercept) for statsmodels
X = sm.add_constant(X)

# 5. Fit the OLS (Ordinary Least Squares) Model
model = sm.OLS(y, X).fit()

# 6. Display the results
print(model.summary())

From the regression, we see the coefficients for most of the variables are statistically significant.  The largest of these are male, number of years of education, and race.  We conclude that hours worked is conditionally associated with the sensitive attributes.

We finally check for conditional outcome differences.  This time we conduct a regression to predict the target variable, income category.

In [ ]:
# 1. Prepare the data
features = ['age', 'sex', 'race', 'num_edu', 'gain', 'loss', 'hpw']
outcome = 'target'

df_reg = df[features + [outcome]].dropna().copy()

# Convert categorical target to binary numeric (0 and 1)
# This assumes '>50K' is the positive class (1)
df_reg['target_numeric'] = (df_reg['target'].str.contains('>50K')).astype(int)

# 2. Encode Categorical Variables
ohe = OneHotEncoder(drop='first', sparse_output=False)
categorical_encoded = ohe.fit_transform(df_reg[['sex', 'race']])
encoded_cols = ohe.get_feature_names_out(['sex', 'race'])
df_encoded = pd.DataFrame(categorical_encoded, columns=encoded_cols, index=df_reg.index)

# 3. Combine numeric and encoded features
X = pd.concat([df_reg[['age', 'num_edu', 'gain', 'loss', 'hpw']], df_encoded], axis=1)

# Use the numeric version of the target
y = df_reg['target_numeric'] 

# 4. Add a constant
X = sm.add_constant(X)

# 5. Fit the Model
# NOTE: If you are predicting a binary outcome (0/1), 
# you should technically use Logit (Logistic Regression) instead of OLS.
model = sm.Logit(y, X).fit() 

# 6. Display
print(model.summary())

In this regression, we see sex and race have high coefficient values that are statistically significant, as does number of years of education.  We conclude that there is a conditional outcome difference of the target variable with respect to the sensitive attributes sex and race.

References$$\begin{array}{l}
\textbf{1. Adult Census Income Dataset} \\
\text{Becker, B., \& Kohavi, R. (1996). } \textit{Adult} \text{ [Dataset]. UCI Machine Learning Repository.} \\
\text{DOI: https://doi.org/10.24432/C5XW20}
\end{array}$$BibTeX Entries (for technical documentation):@misc{adult_1996,
  author       = {Becker, Barry and Kohavi, Ronny},
  title        = {{Adult}},
  year         = {1996},
  howpublished = {UCI Machine Learning Repository},
  note         = {DOI: [https://doi.org/10.24432/C5XW20](https://doi.org/10.24432/C5XW20)}
}
